In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from pathlib import Path
from sklearn.model_selection import train_test_split

# Корневая папка проекта
BASE_DIR = Path.cwd()

if not (BASE_DIR / "data" / "processed" / "sms_clean.csv").exists():
    BASE_DIR = BASE_DIR.parent

# Загружаем очищенный датасет
DATA_PATH = BASE_DIR / "data" / "processed" / "sms_clean.csv"

df = pd.read_csv(DATA_PATH)

# Сбрасываем индекс
df = df.reset_index(drop=True)

print("=== SMS GUARD — ЗАДАНИЕ №6 ===")

print("\nКоличество сообщений:", len(df))

print("\nРаспределение классов:")
print(df["label"].value_counts())

print("\nПервые 5 записей:")
display(df.head())

=== SMS GUARD — ЗАДАНИЕ №6 ===

Количество сообщений: 5103

Распределение классов:
label
0    4501
1     602
Name: count, dtype: int64

Первые 5 записей:


,id,text_raw,text_clean,label
0,1,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,0
1,2,Ok lar... Joking wif u oni...,ok lar joking wif u oni,0
2,3,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in <num> a wkly comp to win fa cup ...,1
3,4,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,0
4,5,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...,0


In [2]:
# ==============================================
# РАЗДЕЛЕНИЕ ДАННЫХ TRAIN / VALIDATION / TEST
# ==============================================

RANDOM_STATE = 42

# Индексы всех сообщений
indices = df.index.to_numpy()

# Первое разделение:
# 70% обучение, 30% временная выборка

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

# Второе разделение:
# 15% валидация, 15% тест

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df.loc[temp_idx, "label"]
)

# Создаём три выборки
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

print("=== РЕЗУЛЬТАТ РАЗДЕЛЕНИЯ ===")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nОбщее количество:")
print(len(train_df) + len(val_df) + len(test_df))

=== РЕЗУЛЬТАТ РАЗДЕЛЕНИЯ ===
Train: 3572
Validation: 765
Test: 766

Общее количество:
5103


In [3]:
# ==============================================
# ПРОВЕРКА РАСПРЕДЕЛЕНИЯ КЛАССОВ
# ==============================================

datasets = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

for name, dataset in datasets.items():

    print(f"\n=== {name} ===")

    print("Количество SMS:", len(dataset))

    print("\nКоличество по классам:")
    print(dataset["label"].value_counts())

    print("\nПроцентное распределение:")
    print(
        (
            dataset["label"]
            .value_counts(normalize=True)
            * 100
        ).round(2)
    )


=== Train ===
Количество SMS: 3572

Количество по классам:
label
0    3151
1     421
Name: count, dtype: int64

Процентное распределение:
label
0    88.21
1    11.79
Name: proportion, dtype: float64

=== Validation ===
Количество SMS: 765

Количество по классам:
label
0    675
1     90
Name: count, dtype: int64

Процентное распределение:
label
0    88.24
1    11.76
Name: proportion, dtype: float64

=== Test ===
Количество SMS: 766

Количество по классам:
label
0    675
1     91
Name: count, dtype: int64

Процентное распределение:
label
0    88.12
1    11.88
Name: proportion, dtype: float64


In [4]:
# ==============================================
# СОХРАНЕНИЕ ИНДЕКСОВ РАЗДЕЛЕНИЯ
# ==============================================

SPLIT_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "sms_split.json"
)

# Проверяем отсутствие пересечений
train_set = set(train_idx)
val_set = set(val_idx)
test_set = set(test_idx)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

# Проверяем, что все записи распределены
assert len(train_set | val_set | test_set) == len(df)

# Подготавливаем данные
split_data = {
    "random_state": RANDOM_STATE,
    "train": train_idx.tolist(),
    "validation": val_idx.tolist(),
    "test": test_idx.tolist()
}

# Создаём папку при необходимости
SPLIT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Сохраняем JSON
with open(SPLIT_PATH, "w", encoding="utf-8") as file:

    json.dump(
        split_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Разделение успешно сохранено!")
print("Путь:", SPLIT_PATH)

print("\nПроверка:")
print("Пересечений между выборками нет.")
print("Всего распределено SMS:", len(df))

Разделение успешно сохранено!
Путь: D:\SMS_Guard_Task_02_Starter\sms_guard\data\processed\sms_split.json

Проверка:
Пересечений между выборками нет.
Всего распределено SMS: 5103
